# Analyze fine-tuning results
This notebook is used to analyze the fine-tuning and the baseline results obtained with Grounding DINO.

### 0. Import libraries and load data

In [ ]:
import json
import polars as pl
import plotly.express as px

RESULTS_PATH = "../../experiments/fine_tuning/"
RESULTS_FOLDER_NAME = "logs_test/"

COLORS = ["#cd968e", "#acb0e0", "#aecbdc", "#bfbfbf", "#bcd5c3"]

In [ ]:
with open(f"{RESULTS_PATH}{RESULTS_FOLDER_NAME}evaluation.json", "r") as f:
    evaluation = json.load(f)

with open(f"{RESULTS_PATH}{RESULTS_FOLDER_NAME}logs.json", "r") as f:
    logs = json.load(f)

### 1. Prepare data and display the statistics

In [ ]:
epochs_info = (
    pl.from_dicts(logs)
    .select(
        "epoch_time", "train_loss", "train_map_50", "train_map_50_95", "val_map_50", "val_map_50_95"
    )
    .with_row_index(name="index", offset=1)
    .with_columns(
        pl.col("epoch_time")
        .str.strptime(pl.Time, format="%H:%M:%S")
        .dt.second()
        .alias("epoch_duration (s)")
    )
    .drop("epoch_time")
    .rename(
        {
            "index": "epoch",
            "train_loss": "train loss",
            "train_map_50": "train mAP@0.5",
            "train_map_50_95": "train mAP@0.5-0.95",
            "val_map_50": "val mAP@0.5",
            "val_map_50_95": "val mAP@0.5-0.95",
        }
    )
)
epochs_info

In [ ]:
fig = px.line(
    epochs_info,
    x="epoch",
    y="train loss",
    title="Evolution of the train loss",
    markers=True,
    color_discrete_sequence=COLORS[0:1],
)
fig.show()

In [ ]:
fig = px.line(
    epochs_info.rename({"train mAP@0.5": "train", "val mAP@0.5": "val"}),
    x="epoch",
    y=["train", "val"],
    labels={"value": "mAP@0.5", "variable": "data set"},
    title="Evolution of mAP@0.5",
    markers=True,
    color_discrete_sequence=COLORS,
)
fig.show()

In [ ]:
fig = px.line(
    epochs_info.rename({"train mAP@0.5-0.95": "train", "val mAP@0.5-0.95": "val"}),
    x="epoch",
    y=["train", "val"],
    labels={"value": "mAP@0.5-0.95", "variable": "data set"},
    title="Evolution of mAP@0.5-0.95",
    markers=True,
    color_discrete_sequence=COLORS,
)
fig.show()

In [ ]:
train_val_metrics = epochs_info[-1].drop("epoch", "train loss", "epoch_duration (s)").to_dicts()[0]
for metric_name, metric_value in train_val_metrics.items():
    print(f"{metric_name}: {metric_value}")
print(f"test mAP@50: {evaluation['test_map_50']}\ntest mAP@50-95: {evaluation['test_map_50_95']}")
print(f"training duration: {epochs_info['epoch_duration (s)'].sum()}s")

###

In [ ]:
map_scores = {
    "mAP@0.5 Desc.": [83.04, 66.35, 65.84],
    "Desc.": [83.04, 66.35, 65.84],
    "mAP@0.5-0.95 Desc.": [78.24, 60.62, 58.98],
    "mAP@0.5 Labels": [89.26, 80.87, 78.87],
    "Labels": [89.26, 80.87, 78.87],
    "mAP@0.5-0.95 Labels": [85.91, 75.26, 72.58],
    "set": ["Train", "Val", "Test"],
}
map_scores_df = pl.from_dict(map_scores)
map_scores_df

In [ ]:
map_scores_melted_df = map_scores_df.melt(
    id_vars=["set"],
    value_vars=["mAP@0.5 Desc.", "mAP@0.5-0.95 Desc.", "mAP@0.5 Labels", "mAP@0.5-0.95 Labels"],
    # value_vars=["Descriptions", "Labels"],
    variable_name="metric_type",
    value_name="score",
)
map_scores_melted_df

In [ ]:
map_scores_melted_df

In [ ]:
fig = px.bar(
    map_scores_melted_df,
    x='set',
    y='score',
    color='metric_type',
    barmode='group',  
    title='Fine-tuning Performance Across Data Splits and Annotation Types',
    labels={
        'set': 'Data Set',
        'score': 'mAP@0.5 (%)',
        'metric_type': 'Annotation Type'
    },
    hover_data=['set', 'metric_type', 'score'], 
    color_discrete_sequence=COLORS
)

font_size = 22
fig.update_layout(
    font=dict(size=font_size),
    title_font_size=font_size,
    xaxis=dict(
        title_font_size=font_size,
        tickfont_size=font_size
    ),
    yaxis=dict(
        title_font_size=font_size,
        tickfont_size=font_size
    ),
    legend=dict(
        font_size=font_size
    ),
    width=1600,  # Set the desired width in pixels
    height=500,  # Set the desired height in pixels
)

fig.show()
fig.write_image("fine_tuning_performance_plot.png", scale=6)


In [ ]:
fig = px.bar(
    map_scores_melted_df,
    x='set',
    y='score',
    color='metric_type',
    barmode='group',  
    title='Fine-tuning Performance Across Data Splits and Annotation Types',
    labels={
        'set': 'Data Set',
        'score': 'mAP (%)',
        'metric_type': 'Metric & Annotation Type'
    },
    hover_data=['set', 'metric_type', 'score'], 
    color_discrete_sequence=COLORS
)

font_size = 28
fig.update_layout(
    font=dict(size=font_size),
    title_font_size=font_size,
    xaxis=dict(
        title_font_size=font_size,
        tickfont_size=font_size
    ),
    yaxis=dict(
        title_font_size=font_size,
        tickfont_size=font_size
    ),
    legend=dict(
        font_size=font_size
    ),
    width=1600,  # Set the desired width in pixels
    height=500,  # Set the desired height in pixels
)

fig.show()
fig.write_image("fine_tuning_performance_plot.png", scale=6)


In [ ]:
import plotly.express as px

# Using the "Sophisticated & Clear" palette for B&W and CVD safety
color_map = {
    "mAP@0.5 Desc.": "#D88C51",       # Muted Amber
    "mAP@0.5-0.95 Desc.": "#F3D8C1",  # Soft Sand
    "mAP@0.5 Labels": "#4A7BA7",      # Sophisticated Steel Blue
    "mAP@0.5-0.95 Labels": "#BDD1E0", # Mist Blue
}

fig = px.bar(
    map_scores_melted_df,
    x="set",
    y="score",
    color="metric_type",
    barmode="group",
    labels={
        "set": "Data Split",
        "score": "mAP (%)",
        "metric_type": "Metric",
    },
    color_discrete_map=color_map,
)

base_font_size = 24

fig.update_layout(
    # --- WHITE BACKGROUND SETTINGS ---
    plot_bgcolor="white",
    paper_bgcolor="white",
    # ---------------------------------
    font=dict(size=base_font_size, color="black"),
    title=None,
    xaxis=dict(
        title_font_size=base_font_size,
        tickfont_size=base_font_size - 2,
        showline=True,
        linewidth=1,
        linecolor='black',
        # # --- TICK SETTINGS ---
        # ticks="outside",
        # tickwidth=1,
        # tickcolor='black',
        # ticklen=5,
        # mirror=True
    ),
    yaxis=dict(
        title_font_size=base_font_size,
        tickfont_size=base_font_size - 2,
        range=[0, 90.3],
        showgrid=True,
        gridcolor='lightgray',
        showline=True,
        linewidth=1,
        linecolor='black',
        # --- TICK SETTINGS ---
        # ticks="outside",
        # tickwidth=1,
        # tickcolor='black',
        # ticklen=5,
        # mirror=True
    ),
    legend=dict(
        title="Metric & Annotation Type",
        font_size=base_font_size - 2,
        x=1.02,
        y=1,
        bgcolor="white",
        bordercolor="black",
        borderwidth=0,
    ),
    width=1200,
    height=450,
    margin=dict(l=80, r=40, t=40, b=70),
)

fig.show()
fig.write_image("fine_tuning_performance_plot.png", scale=4)